In [8]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model="claude-haiku-4-5"


In [9]:
tools = [
    {
        "name": "lookup_order",
        "description": "Look up an order by its order ID and return its item, status, order date, and total. Call this whenever the customer references an order number or asks about the state of an existing order.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID, e.g. A1001"
                }
            },
            "required": ["order_id"]
        }
    },
    {
        # Structured output demo: this tool's only job is to shape Claude's
        # output. We mark it `strict: True` so the API guarantees the
        # returned input matches the schema exactly — every field present,
        # enums honored, no extra keys — no parsing/regex needed on our side.
        "name": "extract_return_request",
        "description": "Call this whenever a customer describes a return or refund request, to capture it as structured data before deciding what to do next. Fill in every field as best you can from the conversation; use null/'unclear' and list missing_information for anything not yet stated.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": ["string", "null"],
                    "description": "The order ID mentioned by the customer, e.g. A1001, or null if they haven't given one."
                },
                "reason": {
                    "type": "string",
                    "enum": [
                        "defective_item",
                        "wrong_item_shipped",
                        "changed_mind",
                        "billing_dispute",
                        "late_delivery",
                        "unclear",
                        "other"
                    ],
                    "description": "The customer's stated reason for the return/refund. Use 'unclear' if not stated clearly enough to classify."
                },
                "urgency": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "How urgent the request sounds, based on tone and content (e.g. angry, time-sensitive)."
                },
                "missing_information": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["order_id", "reason", "item_condition", "preferred_resolution"]
                    },
                    "description": "Which pieces of information the customer has not yet provided."
                },
                "summary": {
                    "type": "string",
                    "description": "One-sentence, neutral summary of what the customer wants."
                }
            },
            "required": ["order_id", "reason", "urgency", "missing_information", "summary"],
            "additionalProperties": False
        }
    }
]

In [10]:
import json
from pathlib import Path

ORDERS_FILE = Path("orders.json")


def lookup_order(order_id: str) -> str:
    """Look up an order by ID in orders.json and return its details as JSON, or an error message."""
    orders = json.loads(ORDERS_FILE.read_text())

    for order in orders:
        if order["order_id"].lower() == order_id.lower():
            return json.dumps(order)

    return f"Error: no order found with ID '{order_id}'."


def process_return_request(order_id, reason, urgency, missing_information, summary) -> str:
    """Apply business rules to an extracted return request and return a JSON
    result describing what should happen next — Claude reads this to decide
    how to phrase its reply, rather than us scripting the reply text here."""
    if not order_id or "order_id" in missing_information:
        return json.dumps({"decision": "need_order_id"})

    orders = json.loads(ORDERS_FILE.read_text())
    order = next((o for o in orders if o["order_id"].lower() == order_id.lower()), None)
    if order is None:
        return json.dumps({"decision": "order_not_found", "order_id": order_id})

    if reason == "unclear":
        decision = "need_clarification"
    elif reason == "billing_dispute" or urgency == "high":
        decision = "escalate_to_specialist"
    else:
        decision = "approve_return"

    return json.dumps({
        "decision": decision,
        "order": order,
        "reason": reason,
        "urgency": urgency,
        "summary": summary,
    })


def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Dispatch a tool_use block to its implementation."""
    if tool_name == "lookup_order":
        return lookup_order(tool_input["order_id"])

    if tool_name == "extract_return_request":
        return process_return_request(**tool_input)

    return f"Error: unknown tool '{tool_name}'."

## Validation and retry

Before we execute a tool call, we validate it in two layers:

1. **Schema validation** — re-check `tool_input` against the tool's own `input_schema` with
   `jsonschema`. `extract_return_request` is already `strict: True`, so the API guarantees this
   layer passes, but `lookup_order` isn't strict, and this also protects us if a tool's schema
   changes later. This is defense-in-depth, not the main event.
2. **Semantic validation** — checks the schema can't express: does `order_id` look like a real
   order ID, is `summary` actually a sentence, are `order_id`/`reason` consistent with
   `missing_information`.

If either layer fails, we don't execute the tool. Instead we send Claude a `tool_result` with
`is_error: True` describing what's wrong, so Claude can retry the call with corrected input on
its next turn — capped at a few attempts per user message so a persistently confused model
doesn't loop forever.

In [11]:
import re
from jsonschema import Draft7Validator

# One schema validator per tool, keyed by name, built once from `tools`.
_SCHEMA_VALIDATORS = {
    tool["name"]: Draft7Validator(tool["input_schema"]) for tool in tools
}

_ORDER_ID_RE = re.compile(r"^[A-Za-z]\d+$")


def validate_schema(tool_name: str, tool_input: dict) -> list[str]:
    """Validate tool_input against the tool's own JSON schema. Returns a list of
    human-readable error strings; empty means valid."""
    validator = _SCHEMA_VALIDATORS.get(tool_name)
    if validator is None:
        return [f"Unknown tool '{tool_name}'."]

    return [error.message for error in validator.iter_errors(tool_input)]


def validate_return_request_semantics(tool_input: dict) -> list[str]:
    """Business-rule checks the JSON schema can't express for extract_return_request."""
    errors = []

    order_id = tool_input.get("order_id")
    missing = tool_input.get("missing_information", [])
    summary = tool_input.get("summary", "")

    if order_id is not None:
        if not _ORDER_ID_RE.match(order_id.strip()):
            errors.append(
                f"order_id '{order_id}' doesn't look like a valid order ID "
                "(expected a letter followed by digits, e.g. A1001)."
            )
        if "order_id" in missing:
            errors.append(
                "order_id is set but 'order_id' is also listed in missing_information — "
                "these are inconsistent."
            )
    elif "order_id" not in missing:
        errors.append("order_id is null but 'order_id' is not listed in missing_information.")

    if not summary.strip():
        errors.append("summary is empty — provide a one-sentence neutral summary.")
    elif len(summary.split()) < 3:
        errors.append("summary is too short to be a meaningful one-sentence summary.")

    return errors


def validate_tool_call(tool_name: str, tool_input: dict) -> list[str]:
    """Run schema validation, then (if that passes) tool-specific semantic validation."""
    errors = validate_schema(tool_name, tool_input)
    if errors:
        return errors

    if tool_name == "extract_return_request":
        return validate_return_request_semantics(tool_input)

    return []

In [12]:
def add_user_message(messages: list, text: str) -> None:
    """Append a user turn to the conversation."""
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages: list, text: str) -> None:
    """Append an assistant turn to the conversation."""
    messages.append({"role": "assistant", "content": text})

In [13]:
SYSTEM_PROMPT = """You are a helpful shop assistant for an online store.

If the customer is just asking about the status of an existing order, call
lookup_order directly.

If the customer describes wanting a return or refund, call
extract_return_request first — even if they haven't given you every detail
yet, since the tool's result will tell you what's missing and what to do
next. Fill in every field of that tool as best you can from the conversation
so far; don't ask the customer questions before calling it.

Follow the extract_return_request result's "decision" field to guide your
reply:
- need_order_id: ask for their order ID.
- order_not_found: let them know that order ID doesn't match anything, and ask them to double-check it.
- need_clarification: ask why they'd like to return the item.
- escalate_to_specialist: tell them you're escalating this to a specialist.
- approve_return: confirm the return and explain next steps.
"""

MAX_VALIDATION_RETRIES = 2

# The whole conversation lives in this list — we resend it on every call.
messages = []

def send_message(messages: list) -> str:
    """Call Claude with the current conversation, running the tool-use loop
    until Claude has no more tools to call, and return the final reply text."""
    # Counts validation failures across the whole turn (all tool calls), so a
    # model that keeps producing invalid input can't loop forever.
    validation_retries = 0

    while True:
        response = client.messages.create(
            model=model,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=tools,
            messages=messages,
        )

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return next(block.text for block in response.content if block.type == "text")

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"[tool call] {block.name}({block.input})")
                errors = validate_tool_call(block.name, block.input)

                if errors and validation_retries < MAX_VALIDATION_RETRIES:
                    validation_retries += 1
                    print(f"[validation failed, retry {validation_retries}/{MAX_VALIDATION_RETRIES}] {errors}")
                    error_text = (
                        "Your tool call was invalid:\n"
                        + "\n".join(f"- {error}" for error in errors)
                        + "\nPlease call the tool again with corrected input."
                    )
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": error_text,
                        "is_error": True,
                    })
                    continue

                if errors:
                    print(f"[validation failed, giving up after {MAX_VALIDATION_RETRIES} retries] {errors}")

                result = execute_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })

        messages.append({"role": "user", "content": tool_results})

In [ ]:
while True:
    try:
        user_input = input("You: ")
        print(f"User: {user_input}")
    except (EOFError, KeyboardInterrupt):
        break

    # VS Code's input box returns "" for both Escape and a blank Enter, so
    # blank input doubles as the way to quit here.
    if not user_input.strip() or user_input.lower() in ("quit", "exit"):
        break

    add_user_message(messages, user_input)
    # send_message appends the assistant turn(s) itself, including any
    # intermediate tool_use/tool_result turns from the tool loop.
    reply = send_message(messages)

    print(f"Assistant: {reply}")

User: Hi want return back my order is #919123blabla
[tool call] lookup_order({'order_id': '#919123blabla'})
Assistant: I wasn't able to find an order with that ID. Could you double-check the order number? It might be listed in your confirmation email or account. Order IDs are usually formatted like "A1001" or similar. 

Once you provide the correct order ID, I can help you process the return.
User: My order is definitly is #919123blabla
Assistant: I understand the order number is #919123blabla, but unfortunately our system didn't find a match for that ID. This can happen for a few reasons:

- There might be a typo in the order number
- It could be formatted differently than expected
- The order might not be in our system yet

Could you:
1. Check your confirmation email for the exact order ID?
2. Or let me know if there are any letters or numbers you might have mixed up?

Once we locate the correct order number, I'll be able to help you process the return right away.
User: I need to spe